# Notebook 2: Model Training
## AI-Generated Face Detection

**Purpose:** Train the dual-branch deepfake detection architecture:
1. **Branch 1:** Fine-tune CLIP ViT-L/14 on RGB face crops
2. **Branch 2:** Train EfficientNet-B0 on DCT frequency representations
3. **Late Fusion:** Train a meta-classifier on concatenated features from both branches

**Prerequisites:** Run Notebook 1 first. Processed dataset should be at:  
`/content/drive/MyDrive/deepfake_detection/processed_dataset/`

**Hardware:** Colab T4 GPU (16GB VRAM)

## 1. Setup & Installation

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
!pip install timm albumentations scipy scikit-learn

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import cv2
import time
import json
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms
import timm
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Debug: Find where Notebook 1 saved the data
import glob

PROJECT_DIR = '/content/drive/MyDrive/deepfake_detection'

print("Checking project directory exists:", os.path.exists(PROJECT_DIR))
print()

# Search for manifest.csv anywhere under the project dir
manifests = glob.glob(os.path.join(PROJECT_DIR, '**', 'manifest.csv'), recursive=True)
print("Found manifest.csv at:")
for m in manifests:
    print(f"  {m}")

if not manifests:
    # Maybe it's at a different top-level path
    print("\nSearching broader...")
    manifests = glob.glob('/content/drive/MyDrive/**/manifest.csv', recursive=True)
    for m in manifests:
        print(f"  {m}")

print("\nContents of project dir:")
if os.path.exists(PROJECT_DIR):
    for item in os.listdir(PROJECT_DIR):
        full = os.path.join(PROJECT_DIR, item)
        print(f"  {'[DIR]' if os.path.isdir(full) else '[FILE]'} {item}")

In [ ]:
# Check what's inside processed_dataset
processed = '/content/drive/MyDrive/deepfake_detection/processed_dataset'

for item in os.listdir(processed):
    full = os.path.join(processed, item)
    if os.path.isdir(full):
        count = len(os.listdir(full))
        print(f"  [DIR] {item} ({count} items)")
    else:
        print(f"  [FILE] {item}")

In [ ]:
import os
from pathlib import Path

crops_dir = '/content/drive/MyDrive/deepfake_detection/processed_dataset/crops'

for folder in sorted(os.listdir(crops_dir)):
    full = os.path.join(crops_dir, folder)
    if os.path.isdir(full):
        files = [f for f in os.listdir(full) if Path(f).suffix.lower() in {'.png', '.jpg', '.jpeg'}]
        print(f"  {folder}: {len(files)} images")

In [ ]:
# Paths
PROJECT_DIR = '/content/drive/MyDrive/deepfake_detection'
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'processed_dataset')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Load manifest
manifest_path = os.path.join(PROCESSED_DIR, 'manifest.csv')
manifest_df = pd.read_csv(manifest_path)

print(f'Total records: {len(manifest_df)}')
print(f'\nSplit distribution:')
print(manifest_df.groupby(['split', 'category']).size().unstack(fill_value=0).to_string())

## 2. Dataset & DataLoader

Reuse the `DeepfakeDataset` class from Notebook 1, now with training augmentations via Albumentations.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Training augmentations (simulates real-world degradation)
# Note: using kwargs that work across Albumentations 1.x and 2.x
try:
    train_augment = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=5, p=0.2),
        A.ImageCompression(quality_range=(60, 95), p=0.5),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.3),
        A.GaussNoise(std_range=(0.04, 0.2), p=0.2),
        A.Downscale(scale_range=(0.5, 0.9), p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
except TypeError:
    # Fallback for older Albumentations (<2.0)
    train_augment = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=5, p=0.2),
        A.ImageCompression(quality_lower=60, quality_upper=95, p=0.5),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.3),
        A.GaussNoise(var_limit=(10, 50), p=0.2),
        A.Downscale(scale_min=0.5, scale_max=0.9, p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

val_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

print('Augmentation pipelines ready.')


In [ ]:
class DeepfakeDataset(Dataset):
    """
    Dataset for loading face crops + DCT representations.
    Returns both RGB and DCT inputs for the dual-branch architecture.
    """
    def __init__(self, manifest_df, data_root, split='train', augment=None):
        self.data_root = data_root
        self.df = manifest_df[manifest_df['split'] == split].reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load RGB crop
        rgb_path = os.path.join(self.data_root, row['crop_path'])
        rgb = np.array(Image.open(rgb_path).convert('RGB'))

        # Apply augmentations
        if self.augment:
            rgb = self.augment(image=rgb)['image']
        else:
            # Default: just normalise and convert to tensor
            rgb = val_transform(image=rgb)['image']

        # Load DCT representation
        dct_path = os.path.join(self.data_root, row['dct_path'])
        dct = np.load(dct_path)
        dct = torch.tensor(dct, dtype=torch.float32).unsqueeze(0)  # (1, H, W)

        label = torch.tensor(row['label'], dtype=torch.float32)

        return rgb, dct, label

# Create datasets
train_ds = DeepfakeDataset(manifest_df, PROCESSED_DIR, split='train', augment=train_augment)
val_ds = DeepfakeDataset(manifest_df, PROCESSED_DIR, split='val', augment=None)

print(f'Train size: {len(train_ds)}')
print(f'Val size:   {len(val_ds)}')

# Quick shape check
rgb, dct, label = train_ds[0]
print(f'\nRGB shape: {rgb.shape}, DCT shape: {dct.shape}, Label: {label.item()}')

In [ ]:
# DataLoaders
# Batch size 16 is safe for T4 with ViT-L/14 + mixed precision
# We use gradient accumulation to simulate larger effective batch size

BATCH_SIZE = 16
ACCUMULATION_STEPS = 4  # Effective batch size = 16 × 4 = 64

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Effective batch size: {BATCH_SIZE * ACCUMULATION_STEPS}')

---
## 3. Branch 1: Vision Transformer (ViT-L/14 CLIP)

**Architecture:**
- Backbone: ViT-L/14 pretrained with CLIP (via `timm`)
- Frozen: Blocks 0–17 (first 18 of 24 blocks)
- Unfrozen: Blocks 18–23 + classification head
- Head: CLS token → Linear(1024, 256) → ReLU → Dropout(0.3) → Linear(256, 1)

**Training:**
- Loss: BCEWithLogitsLoss (label smoothing 0.05)
- Optimiser: AdamW (lr=2e-5, cosine annealing)
- Mixed precision (FP16)
- Early stopping (patience=3 on val AUC)

In [ ]:
class ViTDeepfakeDetector(nn.Module):
    """
    ViT-L/14 CLIP backbone with binary classification head.
    Only the last 6 transformer blocks and head are trainable.
    """
    def __init__(self, freeze_blocks=18):
        super().__init__()

        # Load pretrained CLIP ViT-L/14
        self.backbone = timm.create_model(
            'vit_large_patch14_clip_224',
            pretrained=True,
            num_classes=0  # Remove original head
        )

        # Freeze early blocks
        for i, block in enumerate(self.backbone.blocks):
            if i < freeze_blocks:
                for param in block.parameters():
                    param.requires_grad = False

        # Also freeze patch embed and pos embed
        for param in self.backbone.patch_embed.parameters():
            param.requires_grad = False
        if hasattr(self.backbone, 'pos_embed') and self.backbone.pos_embed is not None:
            self.backbone.pos_embed.requires_grad = False
        if hasattr(self.backbone, 'cls_token') and self.backbone.cls_token is not None:
            self.backbone.cls_token.requires_grad = False

        # Classification head
        feat_dim = self.backbone.num_features  # 1024 for ViT-L
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        features = self.backbone(x)  # CLS token output
        return self.head(features)

    def extract_features(self, x):
        """Extract penultimate features (256-dim) for late fusion."""
        features = self.backbone(x)
        # Pass through first layer of head only
        x = self.head[0](features)  # Linear(1024, 256)
        x = self.head[1](x)         # ReLU
        return x

# Create model
vit_model = ViTDeepfakeDetector(freeze_blocks=18).to(device)

# Count parameters
total_params = sum(p.numel() for p in vit_model.parameters())
trainable_params = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params / 1e6:.1f}M')
print(f'Trainable parameters: {trainable_params / 1e6:.1f}M')
print(f'Frozen parameters:    {(total_params - trainable_params) / 1e6:.1f}M')

In [ ]:
# Label smoothing BCE loss
class LabelSmoothingBCELoss(nn.Module):
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return self.bce(logits, targets)

# Training setup
NUM_EPOCHS = 15
LR = 2e-5
PATIENCE = 3

criterion = LabelSmoothingBCELoss(smoothing=0.05)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, vit_model.parameters()),
    lr=LR, weight_decay=0.01, betas=(0.9, 0.999)
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = GradScaler()

print(f'Optimiser: AdamW (lr={LR}, wd=0.01)')
print(f'Scheduler: CosineAnnealing → 1e-6')
print(f'Epochs: {NUM_EPOCHS} (early stopping patience={PATIENCE})')

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, accumulation_steps, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    optimizer.zero_grad()
    pending = False  # FIX: track whether an unflushed backward exists

    for i, (rgb, dct, labels) in enumerate(tqdm(loader, desc='Training')):
        rgb = rgb.to(device)
        labels = labels.to(device)

        with autocast():
            logits = model(rgb).squeeze(1)
            loss = criterion(logits, labels) / accumulation_steps

        scaler.scale(loss).backward()
        pending = True

        if (i + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            pending = False

        running_loss += loss.item() * accumulation_steps
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.cpu().numpy())

    # FIX: only flush if there is actually a pending backward
    if pending:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    avg_loss = running_loss / len(loader)
    auc = roc_auc_score(all_labels, all_preds)
    acc = accuracy_score(all_labels, [1 if p > 0.5 else 0 for p in all_preds])
    return avg_loss, auc, acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for rgb, dct, labels in tqdm(loader, desc='Evaluating'):
        rgb = rgb.to(device)
        labels = labels.to(device)

        with autocast():
            logits = model(rgb).squeeze(1)
            loss = criterion(logits, labels)

        running_loss += loss.item()
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader)
    auc = roc_auc_score(all_labels, all_preds)
    acc = accuracy_score(all_labels, [1 if p > 0.5 else 0 for p in all_preds])
    return avg_loss, auc, acc, all_preds, all_labels

print('Training and evaluation functions ready.')


In [ ]:
# ============================================
# TRAIN ViT BRANCH
# ============================================

print('=' * 60)
print('TRAINING BRANCH 1: Vision Transformer (ViT-L/14 CLIP)')
print('=' * 60)

best_val_auc = 0.0
patience_counter = 0
vit_history = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': [], 'lr': []}

for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    current_lr = optimizer.param_groups[0]['lr']

    train_loss, train_auc, train_acc = train_one_epoch(
        vit_model, train_loader, criterion, optimizer, scaler, ACCUMULATION_STEPS, device
    )
    val_loss, val_auc, val_acc, _, _ = evaluate(vit_model, val_loader, criterion, device)

    scheduler.step()
    elapsed = time.time() - start_time

    vit_history['train_loss'].append(train_loss)
    vit_history['val_loss'].append(val_loss)
    vit_history['train_auc'].append(train_auc)
    vit_history['val_auc'].append(val_auc)
    vit_history['lr'].append(current_lr)

    print(f'\nEpoch {epoch+1}/{NUM_EPOCHS} ({elapsed:.0f}s) | LR: {current_lr:.2e}')
    print(f'  Train - Loss: {train_loss:.4f} | AUC: {train_auc:.4f} | Acc: {train_acc:.4f}')
    print(f'  Val   - Loss: {val_loss:.4f} | AUC: {val_auc:.4f} | Acc: {val_acc:.4f}')

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': vit_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auc': val_auc,
            'val_loss': val_loss,
        }, os.path.join(CHECKPOINT_DIR, 'vit_best.pth'))
        print(f'  New best model saved (AUC: {val_auc:.4f})')
    else:
        patience_counter += 1
        print(f'  No improvement ({patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping triggered at epoch {epoch+1}')
            break

# FIX: reload best weights so any downstream eval uses the best model, not the last
vit_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'vit_best.pth'), map_location=device, weights_only=False)
vit_model.load_state_dict(vit_ckpt['model_state_dict'])
print(f'\nViT training complete. Best val AUC: {best_val_auc:.4f} (weights reloaded)')


In [ ]:
# Plot ViT training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(vit_history['train_loss']) + 1)

axes[0].plot(epochs_range, vit_history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs_range, vit_history['val_loss'], 'r-o', label='Val')
axes[0].set_title('ViT Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, vit_history['train_auc'], 'b-o', label='Train')
axes[1].plot(epochs_range, vit_history['val_auc'], 'r-o', label='Val')
axes[1].set_title('ViT AUC-ROC')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, vit_history['lr'], 'g-o')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Branch 1: ViT Training History', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'vit_training_history.png'), dpi=150)
plt.show()

---
## 4. Branch 2: Frequency Analysis CNN (EfficientNet-B0)

**Architecture:**
- Backbone: EfficientNet-B0 (ImageNet pretrained)
- Modified first conv layer for single-channel DCT input
- Head: same structure as ViT head (→ 256 → 1)

**Trains much faster** than ViT due to smaller model and single-channel input.

In [ ]:
class FrequencyBranchDetector(nn.Module):
    """
    EfficientNet-B0 adapted for single-channel DCT input.
    First conv weights are averaged across RGB channels.
    """
    def __init__(self):
        super().__init__()

        # Load pretrained EfficientNet-B0
        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=True,
            num_classes=0  # Remove original head
        )

        # Modify first conv layer for 1-channel input
        old_conv = self.backbone.conv_stem
        new_conv = nn.Conv2d(
            1, old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=old_conv.bias is not None
        )
        # Average pretrained weights across channel dimension
        with torch.no_grad():
            new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
            if old_conv.bias is not None:
                new_conv.bias.copy_(old_conv.bias)
        self.backbone.conv_stem = new_conv

        # Classification head
        feat_dim = self.backbone.num_features  # 1280 for EfficientNet-B0
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

    def extract_features(self, x):
        """Extract penultimate features (256-dim) for late fusion."""
        features = self.backbone(x)
        x = self.head[0](features)  # Linear
        x = self.head[1](x)         # ReLU
        return x

# Create model
freq_model = FrequencyBranchDetector().to(device)

total_params = sum(p.numel() for p in freq_model.parameters())
print(f'Frequency branch parameters: {total_params / 1e6:.1f}M (all trainable)')

In [ ]:
# Frequency branch training setup
FREQ_EPOCHS = 20
FREQ_LR = 1e-4
FREQ_PATIENCE = 4

freq_criterion = LabelSmoothingBCELoss(smoothing=0.05)
freq_optimizer = optim.AdamW(freq_model.parameters(), lr=FREQ_LR, weight_decay=0.01)
freq_scheduler = optim.lr_scheduler.CosineAnnealingLR(freq_optimizer, T_max=FREQ_EPOCHS, eta_min=1e-6)
freq_scaler = GradScaler()

print(f'Frequency branch training config:')
print(f'  LR: {FREQ_LR}, Epochs: {FREQ_EPOCHS}, Patience: {FREQ_PATIENCE}')

In [ ]:
# Frequency branch train/eval functions (use DCT input instead of RGB)

def train_freq_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for rgb, dct, labels in tqdm(loader, desc='Training (Freq)'):
        dct = dct.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        with autocast():
            logits = model(dct).squeeze(1)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        all_preds.extend(torch.sigmoid(logits).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader)
    auc = roc_auc_score(all_labels, all_preds)
    acc = accuracy_score(all_labels, [1 if p > 0.5 else 0 for p in all_preds])
    return avg_loss, auc, acc


@torch.no_grad()
def eval_freq(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for rgb, dct, labels in tqdm(loader, desc='Evaluating (Freq)'):
        dct = dct.to(device)
        labels = labels.to(device)

        with autocast():
            logits = model(dct).squeeze(1)
            loss = criterion(logits, labels)

        running_loss += loss.item()
        all_preds.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader)
    auc = roc_auc_score(all_labels, all_preds)
    acc = accuracy_score(all_labels, [1 if p > 0.5 else 0 for p in all_preds])
    return avg_loss, auc, acc, all_preds, all_labels

print('Frequency branch training functions ready.')

In [ ]:
# ============================================
# TRAIN FREQUENCY BRANCH
# ============================================

print('=' * 60)
print('TRAINING BRANCH 2: Frequency Analysis (EfficientNet-B0 + DCT)')
print('=' * 60)

best_freq_auc = 0.0
freq_patience_counter = 0
freq_history = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': []}

for epoch in range(FREQ_EPOCHS):
    start_time = time.time()

    train_loss, train_auc, train_acc = train_freq_epoch(
        freq_model, train_loader, freq_criterion, freq_optimizer, freq_scaler, device
    )
    val_loss, val_auc, val_acc, _, _ = eval_freq(
        freq_model, val_loader, freq_criterion, device
    )

    freq_scheduler.step()
    elapsed = time.time() - start_time

    freq_history['train_loss'].append(train_loss)
    freq_history['val_loss'].append(val_loss)
    freq_history['train_auc'].append(train_auc)
    freq_history['val_auc'].append(val_auc)

    print(f'\nEpoch {epoch+1}/{FREQ_EPOCHS} ({elapsed:.0f}s)')
    print(f'  Train - Loss: {train_loss:.4f} | AUC: {train_auc:.4f} | Acc: {train_acc:.4f}')
    print(f'  Val   - Loss: {val_loss:.4f} | AUC: {val_auc:.4f} | Acc: {val_acc:.4f}')

    if val_auc > best_freq_auc:
        best_freq_auc = val_auc
        freq_patience_counter = 0
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': freq_model.state_dict(),
            'val_auc': val_auc,
        }, os.path.join(CHECKPOINT_DIR, 'freq_best.pth'))
        print(f'  New best model saved (AUC: {val_auc:.4f})')
    else:
        freq_patience_counter += 1
        print(f'  No improvement ({freq_patience_counter}/{FREQ_PATIENCE})')
        if freq_patience_counter >= FREQ_PATIENCE:
            print(f'\nEarly stopping at epoch {epoch+1}')
            break

# FIX: reload best weights
freq_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'freq_best.pth'), map_location=device, weights_only=False)
freq_model.load_state_dict(freq_ckpt['model_state_dict'])
print(f'\nFrequency branch training complete. Best val AUC: {best_freq_auc:.4f} (weights reloaded)')


In [ ]:
# Plot frequency branch training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(freq_history['train_loss']) + 1)

axes[0].plot(epochs_range, freq_history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs_range, freq_history['val_loss'], 'r-o', label='Val')
axes[0].set_title('Frequency Branch Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, freq_history['train_auc'], 'b-o', label='Train')
axes[1].plot(epochs_range, freq_history['val_auc'], 'r-o', label='Val')
axes[1].set_title('Frequency Branch AUC-ROC')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Branch 2: EfficientNet-B0 + DCT Training History', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'freq_training_history.png'), dpi=150)
plt.show()

---
## 5. Late Fusion: Meta-Classifier

1. Load best checkpoints for both branches
2. Extract 256-dim penultimate features from each branch
3. Concatenate → 512-dim feature vector
4. Train a small MLP meta-classifier
5. Apply temperature scaling for probability calibration

In [ ]:
# Load best models
print('Loading best checkpoints...')

# Using weights_only=False to allow loading of metadata/scalars in the checkpoint
vit_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'vit_best.pth'), map_location=device, weights_only=False)
vit_model.load_state_dict(vit_ckpt['model_state_dict'])
print(f'  ViT:  Epoch {vit_ckpt["epoch"]}, Val AUC: {vit_ckpt["val_auc"]:.4f}')

freq_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'freq_best.pth'), map_location=device, weights_only=False)
freq_model.load_state_dict(freq_ckpt['model_state_dict'])
print(f'  Freq: Epoch {freq_ckpt["epoch"]}, Val AUC: {freq_ckpt["val_auc"]:.4f}')

# Freeze both models
vit_model.eval()
freq_model.eval()
for p in vit_model.parameters(): p.requires_grad = False
for p in freq_model.parameters(): p.requires_grad = False

print('\nBoth models loaded and frozen for feature extraction.')

In [ ]:
# FIX: extract training features from a NON-augmented loader.
# The branches saw augmentations during training, but the meta-classifier
# should learn on clean features representative of the true data distribution.

train_ds_clean = DeepfakeDataset(manifest_df, PROCESSED_DIR, split='train', augment=None)
train_loader_clean = DataLoader(
    train_ds_clean, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

@torch.no_grad()
def extract_all_features(vit_model, freq_model, loader, device):
    vit_features = []
    freq_features = []
    labels_list = []

    for rgb, dct, labels in tqdm(loader, desc='Extracting features'):
        rgb = rgb.to(device)
        dct = dct.to(device)

        with autocast():
            vit_feat = vit_model.extract_features(rgb)    # (B, 256)
            freq_feat = freq_model.extract_features(dct)  # (B, 256)

        vit_features.append(vit_feat.float().cpu())
        freq_features.append(freq_feat.float().cpu())
        labels_list.append(labels)

    vit_features = torch.cat(vit_features, dim=0)
    freq_features = torch.cat(freq_features, dim=0)
    labels = torch.cat(labels_list, dim=0)

    fused = torch.cat([vit_features, freq_features], dim=1)  # (N, 512)
    return fused, labels

print('Extracting training features (no augmentation)...')
train_features, train_labels = extract_all_features(vit_model, freq_model, train_loader_clean, device)

print('Extracting validation features...')
val_features, val_labels = extract_all_features(vit_model, freq_model, val_loader, device)

print(f'\nTrain features shape: {train_features.shape}')
print(f'Val features shape:   {val_features.shape}')


In [ ]:
# Meta-classifier MLP

class MetaClassifier(nn.Module):
    """Small MLP that fuses features from both branches."""
    def __init__(self, input_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        self.temperature = nn.Parameter(torch.ones(1))  # For calibration

    def forward(self, x):
        return self.net(x)

    def predict_calibrated(self, x):
        logits = self.net(x)
        return torch.sigmoid(logits / self.temperature)

meta_model = MetaClassifier(input_dim=512).to(device)
meta_optimizer = optim.Adam(meta_model.parameters(), lr=1e-3)
meta_criterion = nn.BCEWithLogitsLoss()

print(f'Meta-classifier parameters: {sum(p.numel() for p in meta_model.parameters())}')

In [ ]:
# Train meta-classifier with per-epoch eval + early stopping

print('Training meta-classifier...')

META_EPOCHS = 100
META_BATCH = 64
META_PATIENCE = 10  # FIX: early stopping on the meta-classifier

from torch.utils.data import TensorDataset

meta_train_ds = TensorDataset(train_features, train_labels)
meta_train_loader = DataLoader(meta_train_ds, batch_size=META_BATCH, shuffle=True)

meta_val_ds = TensorDataset(val_features, val_labels)
meta_val_loader = DataLoader(meta_val_ds, batch_size=META_BATCH, shuffle=False)

best_meta_auc = 0.0
meta_patience_counter = 0

for epoch in range(META_EPOCHS):
    # Train
    meta_model.train()
    for feats, labs in meta_train_loader:
        feats, labs = feats.to(device), labs.to(device)
        logits = meta_model(feats).squeeze(1)
        loss = meta_criterion(logits, labs)
        meta_optimizer.zero_grad()
        loss.backward()
        meta_optimizer.step()

    # FIX: evaluate EVERY epoch so we don't miss the true peak
    meta_model.eval()
    all_preds, all_labs = [], []
    with torch.no_grad():
        for feats, labs in meta_val_loader:
            feats = feats.to(device)
            probs = torch.sigmoid(meta_model(feats).squeeze(1)).cpu().numpy()
            all_preds.extend(probs)
            all_labs.extend(labs.numpy())
    auc = roc_auc_score(all_labs, all_preds)
    acc = accuracy_score(all_labs, [1 if p > 0.5 else 0 for p in all_preds])

    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1}: Val AUC={auc:.4f}, Acc={acc:.4f}')

    if auc > best_meta_auc:
        best_meta_auc = auc
        meta_patience_counter = 0
        torch.save(meta_model.state_dict(), os.path.join(CHECKPOINT_DIR, 'meta_best.pth'))
    else:
        meta_patience_counter += 1
        if meta_patience_counter >= META_PATIENCE:
            print(f'  Early stopping at epoch {epoch+1} (no improvement for {META_PATIENCE} epochs)')
            break

print(f'\nMeta-classifier training complete. Best val AUC: {best_meta_auc:.4f}')


In [ ]:
# Temperature scaling (calibration)
# Optimise temperature on validation set

print('Calibrating temperature scaling...')

meta_model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'meta_best.pth'), map_location=device))

# Freeze everything except temperature
for name, param in meta_model.named_parameters():
    if name != 'temperature':
        param.requires_grad = False
    else:
        param.requires_grad = True

temp_optimizer = optim.LBFGS([meta_model.temperature], lr=0.01, max_iter=50)
nll_criterion = nn.BCEWithLogitsLoss()

all_val_feats = val_features.to(device)
all_val_labs = val_labels.to(device)

def closure():
    temp_optimizer.zero_grad()
    logits = meta_model.net(all_val_feats).squeeze(1)
    scaled_logits = logits / meta_model.temperature
    loss = nll_criterion(scaled_logits, all_val_labs)
    loss.backward()
    return loss

temp_optimizer.step(closure)

print(f'Optimal temperature: {meta_model.temperature.item():.4f}')

# Save final calibrated model
torch.save(meta_model.state_dict(), os.path.join(CHECKPOINT_DIR, 'meta_calibrated.pth'))
print('Calibrated meta-classifier saved.')

---
## 6. Summary & Comparison

In [ ]:
# Quick comparison: ViT alone vs Freq alone vs Fused

# ViT alone on val
vit_model.eval()
_, vit_val_auc, vit_val_acc, _, _ = evaluate(vit_model, val_loader, criterion, device)

# Freq alone on val
freq_model.eval()
_, freq_val_auc, freq_val_acc, _, _ = eval_freq(freq_model, val_loader, freq_criterion, device)

# Fused on val
meta_model.eval()
with torch.no_grad():
    fused_preds = torch.sigmoid(meta_model(val_features.to(device)).squeeze(1)).cpu().numpy()
fused_auc = roc_auc_score(val_labels.numpy(), fused_preds)
fused_acc = accuracy_score(val_labels.numpy(), [1 if p > 0.5 else 0 for p in fused_preds])

print('\n' + '=' * 60)
print('VALIDATION SET COMPARISON')
print('=' * 60)
print(f'{"Model":<25s} {"AUC":>8s} {"Accuracy":>10s}')
print('-' * 45)
print(f'{"ViT-L/14 (Branch 1)":<25s} {vit_val_auc:>8.4f} {vit_val_acc:>10.4f}')
print(f'{"EfficientNet-B0 DCT (B2)":<25s} {freq_val_auc:>8.4f} {freq_val_acc:>10.4f}')
print(f'{"Late Fusion (Combined)":<25s} {fused_auc:>8.4f} {fused_acc:>10.4f}')
print('=' * 45)

In [ ]:
# Save training history and results
results = {
    'vit_best_val_auc': best_val_auc,
    'freq_best_val_auc': best_freq_auc,
    'fused_val_auc': fused_auc,
    'fused_val_acc': fused_acc,
    'temperature': meta_model.temperature.item(),
    'vit_history': vit_history,
    'freq_history': freq_history,
}

with open(os.path.join(CHECKPOINT_DIR, 'training_results.json'), 'w') as f:
    json.dump(results, f, indent=2, default=str)

print('\n' + '=' * 60)
print('NOTEBOOK 2 COMPLETE')
print('=' * 60)
print(f'\nCheckpoints saved to: {CHECKPOINT_DIR}')
print(f'  • vit_best.pth')
print(f'  • freq_best.pth')
print(f'  • meta_calibrated.pth')
print(f'  • training_results.json')
print(f'  • vit_training_history.png')
print(f'  • freq_training_history.png')
print(f'\nReady for Notebook 3: Evaluation & Deployment')